# Normalizzazione dataset film

| File | Contenuto | Chiave |
|---|---|---|
| `film.csv` | un film per riga (MovieID, Title, Year) | `MovieID` |
| `genres.csv` | un genere per riga, senza ripetizioni | `generi_id` |
| `generi_film.csv` | tabella associativa film ↔ generi | (`MovieID`, `generi_id`) |

## Caricamento dati

In [22]:
import pandas as pd

In [23]:
fp = r"..\01_dati_originali\Elenco Movies definitivo.csv"

df_film = pd.read_csv(fp)

df_film.head()

,MovieID,Title,Genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


## Controllo nan e duplicati

In [24]:
df_film.isna().sum()

MovieID    0
Title      0
Genres     0
dtype: int64

In [25]:
df_film_unique = df_film.drop_duplicates()

print(f"Numero righe dataframe con duplicati: {df_film.shape[0]}")
print(f"Numero righe dataframe senza duplicati: {df_film_unique.shape[0]}")

Numero righe dataframe con duplicati: 3883
Numero righe dataframe senza duplicati: 3883


In [26]:
# MovieID sarà la chiave primaria di film.csv: deve essere unico
assert df_film['MovieID'].is_unique, "MovieID duplicati!"

## Controllo errori di formattazione

In [27]:
t = df_film['Title']

controlli = {
    'spazi_doppi':       t.str.contains(r'\s{2,}'),
    'spazi_bordi':       t.str.contains(r'^\s|\s$'),
    'encoding_rotto':    t.str.contains(r'Ã|Â|�'),         # es. "CitÃ©" invece di "Cité"
    'parentesi_aperte':  t.str.count(r'\(') != t.str.count(r'\)'),
    'inizia_minuscola':  t.str.match(r'^[a-z]'),
    'punteggiatura_dop': t.str.contains(r'[,.;:!?]{2,}') & ~t.str.contains(r'\.\.\.'),
    'spazio_prima_punt': t.str.contains(r'\s[,;:!?]'),
}

for nome, mask in controlli.items():
    print(f"{nome}: {mask.sum()}")
    if mask.any():
        display(df_film.loc[mask, ['MovieID', 'Title']].head(10))

spazi_doppi: 0
spazi_bordi: 0
encoding_rotto: 0
parentesi_aperte: 0
inizia_minuscola: 1


,MovieID,Title
2531,2600,eXistenZ (1999)


punteggiatura_dop: 3


,MovieID,Title
713,722,"Haunted World of Edward D. Wood Jr., The (1995)"
2026,2095,"Shaggy D.A., The (1976)"
3798,3868,"Naked Gun: From the Files of Police Squad!, Th..."


spazio_prima_punt: 0


## Estrazione anno

In [28]:
df_film['Year'] = df_film['Title'].str.extract(r'\((\d{4})\)\s*$', expand=False).astype('Int64')

df_film.head()

,MovieID,Title,Genres,Year
0,1,Toy Story (1995),Animation|Children's|Comedy,1995
1,2,Jumanji (1995),Adventure|Children's|Fantasy,1995
2,3,Grumpier Old Men (1995),Comedy|Romance,1995
3,4,Waiting to Exhale (1995),Comedy|Drama,1995
4,5,Father of the Bride Part II (1995),Comedy,1995


In [29]:
df_film[df_film['Year'].isna()]

,MovieID,Title,Genres,Year
15,16,Casino (not_def),Drama|Thriller,<NA>


In [30]:
df_film.iloc[15,3] = 1995

df_film.isna().sum()

MovieID    0
Title      0
Genres     0
Year       0
dtype: int64

## Pulizia titolo

In [31]:
df_film['Title'] = (
    df_film['Title']
      .str.replace(r'\s*\(\d{4}(?:-\d{4})?\)\s*$', '', regex=True)
      .str.strip()
      .mask(lambda s: s == '')  # titolo composto solo dall'anno, es. MovieID 3936 "(1943)"
)

df_film.head()

,MovieID,Title,Genres,Year
0,1,Toy Story,Animation|Children's|Comedy,1995
1,2,Jumanji,Adventure|Children's|Fantasy,1995
2,3,Grumpier Old Men,Comedy|Romance,1995
3,4,Waiting to Exhale,Comedy|Drama,1995
4,5,Father of the Bride Part II,Comedy,1995


In [32]:
df_film.loc[df_film['Title'].isna()]

,MovieID,Title,Genres,Year
3866,3936,NaN,Drama|Thriller,1943


In [33]:
df_film.loc[df_film['MovieID'] == 3936, 'Title'] = "Phantom of the Opera, The"

## Estrazione genere

In [34]:
# Refuso nel separatore: "Comedy--Horror" (Scary Movie) deve essere "Comedy|Horror"
print(df_film.loc[df_film['Genres'].str.contains('--', regex=False), ['MovieID', 'Title', 'Genres']])

df_film['Genres'] = df_film['Genres'].str.replace('--', '|', regex=False)

assert not df_film['Genres'].str.contains('--', regex=False).any()

      MovieID        Title          Genres
3716     3785  Scary Movie  Comedy--Horror


In [35]:
df_esploso = (
    df_film.assign(Genres=df_film['Genres'].str.split('|'))
      .explode('Genres')
      .reset_index(drop=True)
)

df_esploso.head()

,MovieID,Title,Genres,Year
0,1,Toy Story,Animation,1995
1,1,Toy Story,Children's,1995
2,1,Toy Story,Comedy,1995
3,2,Jumanji,Adventure,1995
4,2,Jumanji,Children's,1995


## Correzione refusi genere Drama

`Dramatic` e `Dramma` sono refusi di `Drama`.

In [36]:
refusi_drama = {'Dramatic': 'Drama', 'Dramma': 'Drama'}

display(df_esploso.loc[df_esploso['Genres'].isin(refusi_drama)])

df_esploso['Genres'] = df_esploso['Genres'].replace(refusi_drama)

# se un film aveva sia "Drama" sia un refuso, ora avrebbe "Drama" due volte
df_esploso = df_esploso.drop_duplicates(subset=['MovieID', 'Genres']).reset_index(drop=True)

assert not df_esploso['Genres'].isin(refusi_drama).any()
df_esploso['Genres'].value_counts()

,MovieID,Title,Genres,Year
892,527,Schindler's List,Dramatic,1993
1031,608,Fargo,Dramma,1996
5397,3317,Wonder Boys,Dramma,2000
5537,3409,Final Destination,Dramatic,2000
5809,3578,Gladiator,Dramatic,2000


Genres
Drama          1603
Comedy         1199
Action          502
Thriller        490
Romance         471
Horror          343
Adventure       283
Sci-Fi          276
Children's      251
Crime           210
War             142
Documentary     127
Musical         114
Mystery         106
Animation       105
Fantasy          68
Western          68
Film-Noir        44
Name: count, dtype: int64

## Creazione delle 3 tabelle

In [37]:
# 1) film: un film per riga, senza la colonna dei generi
film = df_film[['MovieID', 'Title', 'Year']]

film.head()

,MovieID,Title,Year
0,1,Toy Story,1995
1,2,Jumanji,1995
2,3,Grumpier Old Men,1995
3,4,Waiting to Exhale,1995
4,5,Father of the Bride Part II,1995


In [38]:
# 2) genres: generi unici con un id numerico
genres = (
    pd.DataFrame({'Genres': sorted(df_esploso['Genres'].unique())})
      .rename_axis('generi_id')
      .reset_index()
)
genres['generi_id'] += 1          # id da 1 invece che da 0

genres

,generi_id,Genres
0,1,Action
1,2,Adventure
2,3,Animation
3,4,Children's
4,5,Comedy
5,6,Crime
6,7,Documentary
7,8,Drama
8,9,Fantasy
9,10,Film-Noir


In [39]:
# 3) generi_film: tabella associativa (MovieID, generi_id)
generi_film = (
    df_esploso[['MovieID', 'Genres']]
      .merge(genres, on='Genres', how='left')
      [['MovieID', 'generi_id']]
)

assert generi_film['generi_id'].notna().all()
assert not generi_film.duplicated().any(), "Coppie film-genere ripetute!"

generi_film.head()

,MovieID,generi_id
0,1,3
1,1,4
2,1,5
3,2,2
4,2,4


## Salvataggio

In [40]:
film.to_csv(r"..\03_dati_preprocessati\film.csv", index=False)
genres.to_csv(r"..\03_dati_preprocessati\genres.csv", index=False)
generi_film.to_csv(r"..\03_dati_preprocessati\generi_film.csv", index=False)

## Verifica: ricostruzione con join

Rileggo i 3 csv dal disco e con due `merge` ricostruisco il formato esploso.

In [41]:
film_csv        = pd.read_csv(r"..\03_dati_preprocessati\film.csv")
genres_csv      = pd.read_csv(r"..\03_dati_preprocessati\genres.csv")
generi_film_csv = pd.read_csv(r"..\03_dati_preprocessati\generi_film.csv")

ricostruito = (
    generi_film_csv
      .merge(film_csv, on='MovieID', how='left')
      .merge(genres_csv, on='generi_id', how='left')
      [['MovieID', 'Title', 'Genres', 'Year']]
)

ricostruito.head(10)

,MovieID,Title,Genres,Year
0,1,Toy Story,Animation,1995
1,1,Toy Story,Children's,1995
2,1,Toy Story,Comedy,1995
3,2,Jumanji,Adventure,1995
4,2,Jumanji,Children's,1995
5,2,Jumanji,Fantasy,1995
6,3,Grumpier Old Men,Comedy,1995
7,3,Grumpier Old Men,Romance,1995
8,4,Waiting to Exhale,Comedy,1995
9,4,Waiting to Exhale,Drama,1995


In [42]:
pd.testing.assert_frame_equal(
    ricostruito.reset_index(drop=True),
    df_esploso.reset_index(drop=True),
    check_dtype=False,
)
print(f"OK: {len(ricostruito)} righe identiche al formato esploso")

OK: 6402 righe identiche al formato esploso


## Riepilogo
- Trovato un anno NaN e corretto il valore
- C'era l'anno nella colonna titolo: estratto in `Year` e rimosso da `Title`
- Genere `Comedy--Horror` (Scary Movie): refuso nel separatore, corretto in `Comedy|Horror`
- Generi `Dramatic` (3 film) e `Dramma` (2 film): refusi di `Drama`, corretti
- Dati salvati in forma normalizzata: `film.csv`, `genres.csv`, `generi_film.csv`
- La join delle 3 tabelle